# GLOF Early Warning & Risk Predictor - Experimental Model Audit

This notebook audits and compares models using **unverified experimental proxy labels**. The labels are reproducible as a 5 km geodesic-distance rule around a local 53-row coordinate list, but that list has no event IDs or verifiable upstream provenance. Therefore, this notebook does not create or save a final flood-risk classifier.

This is a student prototype, not an official flood-warning system.

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display

from glof_experiment import (
    MODEL_VARIANTS,
    VARIANT_DESCRIPTIONS,
    add_stability_to_metrics,
    audit_summary,
    make_master_split,
    prepare_model_ready,
    run_geographic_stability,
    run_holdout_experiments,
    save_artifacts,
    set_global_seed,
    split_summary,
)

ROOT = Path.cwd()
set_global_seed(42)
print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"TensorFlow: {tf.__version__}")

Python: 3.12.10
pandas: 3.0.5
scikit-learn: 1.9.0
TensorFlow: 2.21.0


## 1. Inspect the untouched source dataset

In [2]:
source = pd.read_csv(ROOT / "glofguard_training_data.csv")

print(f"Rows: {len(source):,}")
print(f"Columns: {len(source.columns)}")
print(f"Duplicate rows: {source.duplicated().sum():,}")
print(f"Duplicate coordinate rows: {source.duplicated(['latitude', 'longitude']).sum():,}")

display(pd.DataFrame({"dtype": source.dtypes.astype(str), "missing": source.isna().sum(), "missing_%": source.isna().mean().mul(100).round(3)}))
display(source["label"].value_counts().sort_index().rename("count").to_frame().assign(percentage=lambda x: x["count"] / len(source) * 100))
display(source.select_dtypes(include=np.number).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

Rows: 8,806
Columns: 13
Duplicate rows: 0
Duplicate coordinate rows: 0


,dtype,missing,missing_%
sample_id,int64,0,0.000
area,float64,0,0.000
longitude,float64,0,0.000
latitude,float64,0,0.000
is_top200,bool,0,0.000
temperature,float64,0,0.000
rainfall,float64,0,0.000
label,int64,0,0.000
elevation,int64,0,0.000
distance_to_nearest_settlement_km,float64,0,0.000


,count,percentage
label,,
0,8455,96.014081
1,351,3.985919


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
sample_id,8806.0,4403.500000,2542.217569,1.000000,89.050000,441.250000,2202.250000,4403.500000,6604.750000,8365.750000,8717.950000,8806.000000
area,8806.0,0.015485,0.079300,0.000009,0.000108,0.000240,0.000695,0.001855,0.008929,0.060359,0.197833,3.930649
longitude,8806.0,74.735997,1.389159,71.149000,71.374751,71.976974,73.732534,75.149693,75.805573,76.499228,77.112929,77.208187
latitude,8806.0,35.744941,0.570838,34.482656,34.557091,34.784739,35.252615,35.832334,36.116426,36.713351,36.827929,36.926687
temperature,8806.0,-3.260794,4.077695,-13.370000,-11.830000,-9.140000,-5.920000,-3.030000,-0.460000,4.270000,6.100000,11.820000
rainfall,8806.0,0.905964,0.390373,0.310000,0.310000,0.460000,0.650000,0.770000,1.020000,1.740000,1.870000,2.060000
label,8806.0,0.039859,0.195639,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
elevation,8806.0,4187.092437,434.591369,2023.000000,3029.150000,3418.000000,3939.000000,4195.000000,4469.750000,4863.000000,5130.000000,5541.000000
distance_to_nearest_settlement_km,8806.0,17.586089,10.154722,0.124049,2.691003,5.529472,9.814807,15.146458,23.599746,35.087915,50.773340,55.637226
annual_area_change_km2_per_year,3195.0,-0.000019,0.000444,-0.009360,-0.001230,-0.000270,0.000000,0.000000,0.000000,0.000210,0.000480,0.009780


## 2. Document label status and create model-ready data

`sample_id` is retained only for prediction tracing. `nearest_settlement_name` is excluded. No value is imputed or scaled in the CSV. The annual growth field remains missing where unavailable; its median imputation is fitted using training rows only inside each model pipeline.

The required label-audit columns are included. `matched_event_id` stays empty because no event identifier exists in the available reference file.

In [3]:
model_ready = prepare_model_ready()
summary = audit_summary(model_ready)
print(json.dumps(summary, indent=2))
print(f"Model-ready CSV: {ROOT / 'glofguard_model_ready.csv'}")
print("Label status:", model_ready["label_provenance_status"].unique().tolist())
print("Label method:", model_ready["label_method"].unique().tolist())
print("Missing event IDs:", model_ready["matched_event_id"].isna().sum())
display(model_ready.head(3))

{
  "rows": 8806,
  "columns": 20,
  "missing_values": {
    "sample_id": 0,
    "area": 0,
    "longitude": 0,
    "latitude": 0,
    "is_top200": 0,
    "temperature": 0,
    "rainfall": 0,
    "label": 0,
    "elevation": 0,
    "distance_to_nearest_settlement_km": 0,
    "annual_area_change_km2_per_year": 5611,
    "growth_data_available": 0,
    "label_source": 0,
    "matched_event_id": 8806,
    "matched_event_name": 8455,
    "match_distance_km": 0,
    "matched_reference_row": 8455,
    "label_method": 0,
    "label_provenance_status": 0,
    "geographic_group_025deg": 0
  },
  "duplicate_rows": 0,
  "duplicate_coordinates": 0,
  "label_counts": {
    "0": 8455,
    "1": 351
  },
  "label_percentages": {
    "0": 96.0141,
    "1": 3.9859
  },
  "growth_rows": 3195,
  "is_top200_reconstructed_from_area_mismatches": 0,
  "geographic_groups": 155
}
Model-ready CSV: C:\Users\Essa Ahmed\Desktop\Ayesha\GLOF\glofguard_model_ready.csv
Label status: ['experimental_proxy_unverified']
La

,sample_id,area,longitude,latitude,is_top200,temperature,rainfall,label,elevation,distance_to_nearest_settlement_km,annual_area_change_km2_per_year,growth_data_available,label_source,matched_event_id,matched_event_name,match_distance_km,matched_reference_row,label_method,label_provenance_status,geographic_group_025deg
0,1,0.950522,74.061891,34.828994,1,0.22,1.74,0,3681,9.746248,NaN,0,pakistan_hazardous_lakes_unique.csv (upstream ...,<NA>,<NA>,75.016761,<NA>,WGS84 geodesic distance to nearest reference c...,experimental_proxy_unverified,139_296
1,2,0.034728,74.086340,34.820079,0,-0.81,1.73,0,3663,7.396567,NaN,0,pakistan_hazardous_lakes_unique.csv (upstream ...,<NA>,<NA>,73.847762,<NA>,WGS84 geodesic distance to nearest reference c...,experimental_proxy_unverified,139_296
2,3,0.019693,74.071990,34.806459,0,-0.81,1.73,0,3656,7.619067,NaN,0,pakistan_hazardous_lakes_unique.csv (upstream ...,<NA>,<NA>,75.816189,<NA>,WGS84 geodesic distance to nearest reference c...,experimental_proxy_unverified,139_296


## 3. Create geographically disjoint partitions

Coordinates are placed in approximately 0.25-degree blocks. No block may appear in more than one of train, validation, and test. The test partition remains untouched while preprocessing and threshold selection use training/validation data only.

In [4]:
partition = make_master_split(model_ready, seed=42)
partition_table = split_summary(model_ready, partition)
display(partition_table)

group_sets = {
    name: set(model_ready.loc[partition == name, "geographic_group_025deg"])
    for name in ["train", "validation", "test"]
}
assert not (group_sets["train"] & group_sets["validation"])
assert not (group_sets["train"] & group_sets["test"])
assert not (group_sets["validation"] & group_sets["test"])
print("Geographic group overlap: 0")

,partition,rows,label_0,label_1,positive_percentage,geographic_groups
0,train,5271,5059,212,4.022007,97
1,validation,1780,1705,75,4.213483,31
2,test,1755,1691,64,3.646724,27


Geographic group overlap: 0


## 4. Train the requested holdout comparisons

- Model A: non-redundant features including coordinates and growth
- Model B: Model A without coordinates
- Model C: core model without growth, using all 8,806 rows
- Model D: growth model using only the 3,195 rows with observed growth

`is_top200` is not a model input because the audit found it is exactly reconstructible from an `area` threshold. Each variant compares class-weighted logistic regression, class-weighted random forest, and a small class-weighted ANN. Model C also includes a `DummyClassifier` baseline. Thresholds maximize F1 on validation data only.

In [5]:
holdout_metrics, test_predictions, fitted_preprocessors = run_holdout_experiments(
    model_ready, partition, seed=42
)
display(
    holdout_metrics[
        ["variant", "algorithm", "precision", "recall", "f1", "pr_auc", "roc_auc", "false_negatives", "false_positives", "threshold"]
    ].sort_values(["variant", "pr_auc"], ascending=[True, False])
)

Training Model A - Logistic Regression


Training Model A - Random Forest


Training Model A - ANN


Training Model B - Logistic Regression
Training Model B - Random Forest


Training Model B - ANN


Training Model C - Logistic Regression
Training Model C - Random Forest


Training Model C - ANN


Training Model D - Logistic Regression
Training Model D - Random Forest


Training Model D - ANN


,variant,algorithm,precision,recall,f1,pr_auc,roc_auc,false_negatives,false_positives,threshold
1,Model A,Random Forest,1.000000,0.359375,0.528736,0.465269,0.676195,41,0,0.682085
2,Model A,ANN,0.920000,0.359375,0.516854,0.432063,0.866989,41,2,0.849322
0,Model A,Logistic Regression,0.400000,0.375000,0.387097,0.339082,0.683675,40,36,0.940315
4,Model B,Random Forest,0.920000,0.359375,0.516854,0.461270,0.643610,41,2,0.477502
5,Model B,ANN,0.827586,0.375000,0.516129,0.354898,0.721919,40,5,0.800743
3,Model B,Logistic Regression,0.406780,0.375000,0.390244,0.327463,0.667024,40,35,0.938184
7,Model C,Random Forest,1.000000,0.359375,0.528736,0.466702,0.697119,41,0,0.848640
8,Model C,ANN,0.960000,0.375000,0.539326,0.429418,0.880368,40,1,0.759575
6,Model C,Logistic Regression,0.380952,0.375000,0.377953,0.353137,0.791664,40,39,0.933908
12,Model C,Dummy,0.000000,0.000000,0.000000,0.036467,0.500000,64,0,0.500000


## 5. Check geographic stability

Three-fold `StratifiedGroupKFold` measures threshold-free PR-AUC and ROC-AUC variability across different held-out geographic regions. Imputation, scaling, and class weights are refitted independently inside every fold.

In [6]:
stability = run_geographic_stability(model_ready, seed=42, n_splits=3)
metrics = add_stability_to_metrics(holdout_metrics, stability)
display(
    metrics[
        ["variant", "algorithm", "pr_auc", "recall", "f1", "geographic_cv_pr_auc_mean", "geographic_cv_pr_auc_std"]
    ].sort_values("geographic_cv_pr_auc_mean", ascending=False)
)

Stability Model A fold 1/3 - Logistic Regression
Stability Model A fold 1/3 - Random Forest


Stability Model A fold 1/3 - ANN


Stability Model A fold 2/3 - Logistic Regression
Stability Model A fold 2/3 - Random Forest


Stability Model A fold 2/3 - ANN


Stability Model A fold 3/3 - Logistic Regression
Stability Model A fold 3/3 - Random Forest


...

Stability Model D fold 2/3 - Logistic Regression
Stability Model D fold 2/3 - Random Forest


Stability Model D fold 2/3 - ANN


Stability Model D fold 3/3 - Logistic Regression
Stability Model D fold 3/3 - Random Forest


Stability Model D fold 3/3 - ANN


,variant,algorithm,pr_auc,recall,f1,geographic_cv_pr_auc_mean,geographic_cv_pr_auc_std
3,Model B,Logistic Regression,0.327463,0.375000,0.390244,0.234881,0.117983
0,Model A,Logistic Regression,0.339082,0.375000,0.387097,0.228551,0.155859
6,Model C,Logistic Regression,0.353137,0.375000,0.377953,0.208436,0.149995
5,Model B,ANN,0.354898,0.375000,0.516129,0.189580,0.121195
8,Model C,ANN,0.429418,0.375000,0.539326,0.182767,0.187795
10,Model D,Random Forest,0.074457,0.000000,0.000000,0.139487,0.071231
2,Model A,ANN,0.432063,0.359375,0.516854,0.132210,0.102862
7,Model C,Random Forest,0.466702,0.359375,0.528736,0.121832,0.061777
9,Model D,Logistic Regression,0.103625,0.037037,0.045455,0.121353,0.085567
1,Model A,Random Forest,0.465269,0.359375,0.528736,0.120372,0.060753


## 6. Save auditable artifacts

In [7]:
preprocessing_metadata = save_artifacts(
    model_ready,
    partition,
    metrics,
    test_predictions,
    stability,
    fitted_preprocessors,
)
print(json.dumps(preprocessing_metadata, indent=2))
print("Saved model_metrics.csv and plots under artifacts/.")
print("Final classifier saved: False")

{
  "selected_experimental_configuration": "Model C|Logistic Regression",
  "selection_basis": "validation PR-AUC, then validation recall and F1",
  "fit_scope": "training partition only",
  "features": [
    "area",
    "temperature",
    "rainfall",
    "elevation",
    "distance_to_nearest_settlement_km",
    "longitude",
    "latitude"
  ],
  "label_status": "experimental_proxy_unverified",
  "classifier_saved": false,
  "classifier_not_saved_reason": "The 53-row coordinate reference has no verifiable event IDs or upstream provenance; the 5 km expansion labels are experimental proxies."
}
Saved model_metrics.csv and plots under artifacts/.
Final classifier saved: False


## 7. Simple experimental findings

In [8]:
ranked = metrics.loc[metrics["algorithm"] != "Dummy"].sort_values(
    ["geographic_cv_pr_auc_mean", "validation_pr_auc"], ascending=False
)
best = ranked.iloc[0]
coordinate_comparison = metrics.loc[
    metrics["variant"].isin(["Model A", "Model B"]),
    ["variant", "algorithm", "pr_auc", "geographic_cv_pr_auc_mean", "geographic_cv_pr_auc_std"],
].sort_values(["algorithm", "variant"])

print("Best experimental configuration by mean geographic CV PR-AUC:")
print(best[["variant", "algorithm", "geographic_cv_pr_auc_mean", "geographic_cv_pr_auc_std", "pr_auc", "recall", "f1"]].to_string())
print("\nCoordinate comparison:")
display(coordinate_comparison)
print("\nInterpretation: these scores measure reproduction of the 5 km proxy labels, not validated future GLOF risk.")
print("No final risk model was saved because upstream label provenance is unresolved.")

Best experimental configuration by mean geographic CV PR-AUC:
variant                                  Model B
algorithm                    Logistic Regression
geographic_cv_pr_auc_mean               0.234881
geographic_cv_pr_auc_std                0.117983
pr_auc                                  0.327463
recall                                     0.375
f1                                      0.390244

Coordinate comparison:


,variant,algorithm,pr_auc,geographic_cv_pr_auc_mean,geographic_cv_pr_auc_std
2,Model A,ANN,0.432063,0.132210,0.102862
5,Model B,ANN,0.354898,0.189580,0.121195
0,Model A,Logistic Regression,0.339082,0.228551,0.155859
3,Model B,Logistic Regression,0.327463,0.234881,0.117983
1,Model A,Random Forest,0.465269,0.120372,0.060753
4,Model B,Random Forest,0.461270,0.108869,0.043102



Interpretation: these scores measure reproduction of the 5 km proxy labels, not validated future GLOF risk.
No final risk model was saved because upstream label provenance is unresolved.
